# Qwen single-agent TEC tool-calling experiment


## 1. Clean Colab setup

This notebook is intended for Google Colab.

- Run the clean setup cell first.
- After it finishes, you must use Runtime -> Restart runtime.
- Then continue from the import check cell.
- This notebook does not rebuild IONEX data; it expects a processed parquet file.


In [ ]:
!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"


IMPORTANT:

After this cell finishes, restart the Colab runtime:

Runtime -> Restart runtime

Do not import transformers before restarting.


## 2. Import check

Do not import scipy/sklearn in this notebook.


In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers Qwen imports OK")


## 3. Clone or update repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("Latest commit:")
subprocess.run(["git", "log", "--oneline", "-3"], check=False)


## 4. Dataset setup

Required processed dataset:

`data/processed/tec_regions_2024_03_hourly.parquet`

This notebook does not rebuild IONEX data. Use `notebooks/01_build_tec_dataset.ipynb` to build it, then copy the parquet here.


In [ ]:
from pathlib import Path
import shutil

DATASET_PATH = PROJECT_DIR / "data" / "processed" / "tec_regions_2024_03_hourly.parquet"
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Expected dataset:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())


If the dataset is in Google Drive, uncomment and edit the next cell.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive/tec_regions_2024_03_hourly.parquet")
# shutil.copy2(DRIVE_PARQUET_PATH, DATASET_PATH)
# print("Copied:", DATASET_PATH.exists(), DATASET_PATH)


## 5. Project imports and dataset registration


In [ ]:
from tec_agents.agents.llm_single_agent import LLMSingleAgent
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"

register_dataset(
    dataset_ref="default",
    path=DATASET_PATH,
    file_format="parquet",
)

summary = get_dataset_summary("default")
summary


## 6. Initialize MCP-like tools


In [ ]:
server = build_local_mcp_server(run_id="qwen_single_agent_colab_smoke")
client = LocalMCPClient(server)

tool_names = client.list_tool_names()
print("Available tools:")
for name in tool_names:
    print("-", name)

required = {
    "tec_get_timeseries",
    "tec_compute_high_threshold",
    "tec_detect_high_intervals",
    "tec_compute_series_stats",
    "tec_compare_stats",
    "tec_compute_stability_thresholds",
    "tec_detect_stable_intervals",
}

print("Missing required tools:", sorted(required - set(tool_names)))
print("Legacy report tool present:", "tec_build_report" in tool_names)

assert "tec_build_report" not in tool_names


## 7. Optional Hugging Face login


In [ ]:
# Optional: use a Hugging Face token for higher rate limits or gated models.
# In Colab, add a secret named HF_TOKEN or HF_read_token.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN") or userdata.get("HF_read_token")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HF token found. Continuing unauthenticated.")
except Exception as exc:
    print("HF login skipped:", exc)


## 8. Load Qwen

Important: the default avoids bitsandbytes 4-bit because Colab may fail with `libnvJitLink.so.13`. Use float16 by default on T4.


In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

# Safer default for Colab T4:
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"

# If your Colab runtime supports bitsandbytes correctly, you can try:
# LOAD_IN_4BIT = True
# TORCH_DTYPE = "bfloat16"

MAX_INPUT_TOKENS = 4096

model = LocalQwenChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)


## 9. Run one high-TEC smoke task


In [ ]:
query = "Find high TEC intervals for midlat_europe in March 2024 with q=0.9"

agent = LLMSingleAgent(
    model=model,
    client=client,
    max_steps=12,
    max_tool_calls=8,
    max_parse_retries=2,
    max_tool_retries=2,
    temperature=0.0,
)

result = agent.run(query)

print("success:", result.success)
print("error_message:", result.error_message)
print("tool_sequence:", result.tool_sequence)
print("parse_error_count:", result.parse_error_count)
print("invalid_json_count:", result.invalid_json_count)
print("unknown_format_count:", result.unknown_format_count)
print("repair_attempt_count:", result.repair_attempt_count)

if hasattr(result, "repeated_tool_call_count"):
    print("repeated_tool_call_count:", result.repeated_tool_call_count)
if hasattr(result, "multi_tool_call_output_count"):
    print("multi_tool_call_output_count:", result.multi_tool_call_output_count)
if hasattr(result, "stalled_loop_detected"):
    print("stalled_loop_detected:", result.stalled_loop_detected)

print("answer:")
print(result.answer)


## 10. Check expected sequence


In [ ]:
expected_sequence = [
    "tec_get_timeseries",
    "tec_compute_high_threshold",
    "tec_detect_high_intervals",
]

sequence_match = result.tool_sequence == expected_sequence
print("expected_sequence:", expected_sequence)
print("actual_sequence:", result.tool_sequence)
print("sequence_match:", sequence_match)


## 11. Inspect raw and cleaned model outputs


In [ ]:
for i, raw in enumerate(getattr(result, "raw_model_outputs", []), start=1):
    print("=" * 80)
    print("RAW MODEL OUTPUT", i)
    print(raw)

if hasattr(result, "cleaned_model_outputs"):
    for i, cleaned in enumerate(result.cleaned_model_outputs, start=1):
        print("=" * 80)
        print("CLEANED MODEL OUTPUT", i)
        print(cleaned)


## 12. Save result JSON


In [ ]:
import json

output_path = PROJECT_DIR / "outputs" / "metrics" / "qwen_single_agent_smoke_colab.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "model_name": MODEL_NAME,
    "task_query": query,
    "result": result.to_dict() if hasattr(result, "to_dict") else result.__dict__,
    "tool_sequence": result.tool_sequence,
    "sequence_match": sequence_match,
    "parse_counters": {
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "unknown_format_count": result.unknown_format_count,
        "repair_attempt_count": result.repair_attempt_count,
    },
    "success": result.success,
}

with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

print("Saved:", output_path)


## 13. Optional download result


In [ ]:
# from google.colab import files
# files.download(str(output_path))
